In [1]:
#packages
import pandas as pd
from surprise import Dataset, Reader, SVD


In [2]:
ratings = pd.read_csv('ratings_clean.csv')
recipes = pd.read_csv('recipes_clean.csv')

reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(
    ratings[['user_id', 'recipe_id', 'rating']],
    reader
)


In [3]:
svd = SVD(random_state=1)
trainset = data.build_full_trainset()
svd.fit(trainset)

In [4]:
#without ingredients
def recommender(userId):
    #recipes that the user has already rated
    rated = ratings[ratings['user_id'] == userId]['recipe_id']
    recipes_r = recipes[~recipes['id'].isin(rated)].copy()

    recipes_r['est'] = recipes_r['id'].apply(lambda x: svd.predict(userId, x, clip=False).est)

    recipes_r = recipes_r.sort_values('est', ascending=False)

    return recipes_r[['name', 'est']].head(5)


recommender(1701315) #101823 868654 1701315


,name,est
4851,fresh tomato and cucumber salad,5.284950
6342,jeni s chocolate ganache,5.271098
1494,blue ribbon carrot cake with buttermilk glaze,5.269738
9243,pumpkin ginger scones with cinnamon chips,5.269339
5127,globally inspired fully loaded rolled mashed ...,5.266752


In [ ]:
def recommender_ingredients(userId, my_ingredients):
    match_scores = recipes['ingredients'].apply(lambda x: sum(i.strip() in x for i in my_ingredients.split(',') if i.strip()))

    sim_scores = list(enumerate(match_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    if sim_scores[0][1] == 0:
        return "No recipes contain any of these ingredients"

    sim_scores = [i for i in sim_scores if i[1]>0]

    recipe_indices = [i[0] for i in sim_scores]

    recipes_c = recipes.iloc[recipe_indices][['name', 'n_ingredients', 'id']]

    recipes_c['matches'] = [i[1] for i in sim_scores]

    recipes_c['est'] = recipes_c['id'].apply(lambda x: svd.predict(userId, x, clip=False).est)

    recipes_c = recipes_c.sort_values(['matches', 'n_ingredients', 'est'], ascending=[False, True, False])

    return recipes_c.head(10)

In [6]:
recommender(1701315)


,name,est
4851,fresh tomato and cucumber salad,5.284950
6342,jeni s chocolate ganache,5.271098
1494,blue ribbon carrot cake with buttermilk glaze,5.269738
9243,pumpkin ginger scones with cinnamon chips,5.269339
5127,globally inspired fully loaded rolled mashed ...,5.266752


In [ ]:
recommender_ingredients(1701315, 'red onion, potato, pear, lemon, lime, beans, egg, lettuce, garlic,')

,name,n_ingredients,id,matches,est
7508,mexican chef salad,16,144969,5,5.023206
5026,garlicky cilantro roasted potato salad,10,31286,5,4.921074
12388,youvarlakia avgolemono greek meatball egg lem...,17,59197,5,4.878390
2687,chipotle mexican grill barbacoa burritos by to...,26,323815,4,5.257164
6113,indian spiced omelet,14,98927,4,5.164782
9406,ragu ratatouille hummus plate ragu,21,524479,4,5.150663
8370,orzo with roasted vegetables barefoot contess...,14,111976,4,5.123360
6591,kittencal s famous greek salad,16,66596,4,5.118398
5518,guacamole salad barefoot contessa ina garten,13,239592,4,5.111737
9135,potato nachos,13,350781,4,5.106086


A